In [1]:
import xarray as xr
import h5py
import pandas as pd
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
from os.path import join
import os
from pathlib import Path
from dateutil.parser import parse

from multiprocessing import Pool
from credit.pbs import get_num_cpus

import traceback

In [2]:
mpas_inits = [
            "2025-06-12T23:55:00",
            "2025-06-13T05:55:00",
            "2025-06-13T23:55:00",
            "2025-06-14T05:55:00",
            "2025-06-14T11:55:00",
            "2025-06-14T17:55:00",
            "2025-06-14T23:55:00",
            "2025-06-15T05:55:00",
            "2025-06-15T11:55:00",
            "2025-06-15T17:55:00",
            "2025-06-15T23:55:00",
            "2025-06-16T05:55:00",
            "2025-06-16T11:55:00",
            "2025-06-16T17:55:00",
            "2025-06-16T23:55:00",
            "2025-06-17T05:55:00",
            "2025-06-17T11:55:00",
            "2025-06-17T17:55:00",
            "2025-06-17T23:55:00",
            "2025-06-18T05:55:00",
            "2025-06-18T11:55:00",
            "2025-06-18T17:55:00",
            "2025-06-18T23:55:00",
            "2025-06-19T05:55:00",
            "2025-06-19T11:55:00",
            "2025-06-19T17:55:00",
            "2025-06-19T23:55:00",
            "2025-06-20T05:55:00",
            "2025-06-20T11:55:00",
            "2025-06-20T17:55:00",
            "2025-06-20T23:55:00",
            "2025-06-21T05:55:00",
            "2025-06-21T11:55:00",
            "2025-06-21T17:55:00",
            "2025-06-21T23:55:00",
            "2025-06-22T05:55:00",
            "2025-06-22T11:55:00",
            "2025-06-22T17:55:00",
            "2025-06-22T23:55:00",
            "2025-06-23T05:55:00",
            "2025-06-23T23:55:00",
            "2025-06-24T05:55:00",
            "2025-06-24T11:55:00",
            "2025-06-24T17:55:00",
            "2025-06-24T23:55:00",
            "2025-06-25T05:55:00",
            "2025-06-25T11:55:00",
            "2025-06-25T17:55:00",
            "2025-06-25T23:55:00",
            "2025-06-26T05:55:00",
            "2025-06-26T11:55:00",
            "2025-06-26T17:55:00",
            "2025-06-26T23:55:00",
            "2025-06-27T05:55:00",
            "2025-06-27T11:55:00",
            "2025-06-27T17:55:00",
            "2025-06-27T23:55:00",
            "2025-06-28T05:55:00",
            "2025-06-28T11:55:00",
            "2025-06-28T17:55:00",
            "2025-06-28T23:55:00",
            "2025-06-29T05:55:00",
            "2025-06-29T11:55:00",
            "2025-06-29T17:55:00",
            "2025-06-29T23:55:00",
            "2025-06-30T05:55:00",
            "2025-06-30T11:55:00",
            "2025-06-30T17:55:00",
            "2025-06-30T23:55:00",
            "2025-07-01T05:55:00",
            "2025-07-01T11:55:00",
            "2025-07-01T17:55:00",
            "2025-07-01T23:55:00",
            "2025-07-02T05:55:00",
            "2025-07-02T11:55:00",
            "2025-07-02T17:55:00",
            "2025-07-02T23:55:00",
            "2025-07-03T05:55:00",
            "2025-07-03T11:55:00",
            "2025-07-03T17:55:00",
            "2025-07-03T23:55:00",
            "2025-07-04T05:55:00",
            "2025-07-04T11:55:00",
            "2025-07-04T17:55:00",
            "2025-07-04T23:55:00",
            "2025-07-05T05:55:00",
            "2025-07-05T11:55:00",
            "2025-07-05T17:55:00",
            "2025-07-05T23:55:00",
            "2025-07-06T05:55:00",
            "2025-07-06T11:55:00",
            "2025-07-06T17:55:00",
            "2025-07-06T23:55:00",
            "2025-07-07T05:55:00",
            "2025-07-07T23:55:00",
            "2025-07-08T05:55:00",
            "2025-07-08T11:55:00",
            "2025-07-08T17:55:00",
            "2025-07-08T23:55:00",
        ]

In [3]:
mpas_inits = [pd.Timestamp(init) + pd.Timedelta(5, "m") for init in mpas_inits]

In [4]:
# go through dataset and see which timesteps have frames within tol on the hour and half hour
# we want something either 55/25 or 05/35

In [5]:
zarr_path = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km_2025.zarr"
valid_init_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/valid_init_times_2025"

timestep = (30, "m")
tolerance = (7, "m")
num_forecast_steps = 24
# start_date, end_date = "2025-06-12", "2025-07-10"

In [6]:
ds = xr.open_dataset(zarr_path)

timestep = pd.Timedelta(*timestep)

# start_date, end_date = pd.Timestamp(start_date), pd.Timestamp(end_date)
# ds = ds.sel(t=slice(start_date, end_date))



In [7]:
%%time
# due to missing data, need to have a different list of valid init times
def check_valid_forecast_times(t_init, timestep, num_forecast_steps):
    time_tolerance = pd.Timedelta(*tolerance)
    target_times = [t_init.values + timestep * step for step in range(1, num_forecast_steps + 1)]
    zarr_times = ds.t.sel(t=target_times, method="nearest")
    within_tol = np.abs(zarr_times.values - np.array(target_times).astype(zarr_times.dtype)) < time_tolerance
    return all(within_tol)

valid_times = []
for t in ds.t:
    if check_valid_forecast_times(t, timestep, num_forecast_steps):
        valid_times.append(t.values)

valid_times_da = xr.DataArray(valid_times, coords={"t": valid_times})



CPU times: user 45.7 s, sys: 102 ms, total: 45.8 s
Wall time: 45.7 s


In [10]:
# save valid times for all rollouts. 2025 is rollout only

timesteps = [10, 20, 30]
steps = [72, 36, 24]

for timestep, num_forecast_steps in zip(timesteps, steps):
    timestep = pd.Timedelta(timestep, "m")
    if timestep.seconds >= 3600:
        filename = f"{timestep.seconds // 3600:02d}h_{num_forecast_steps:02}step.nc"
    else:
        filename = f"{timestep.seconds // 60:02d}m_{num_forecast_steps:02}step.nc"
    print(filename)
    valid_init_filepath = join(valid_init_dir, filename)

    valid_times_da.to_netcdf(valid_init_filepath)

10m_72step.nc
20m_36step.nc
30m_24step.nc


In [ ]:
# check which of the mpas inits have a corresponding valid forecast

In [20]:
valid_mpas_inits = []
valid_valid_inits = []
deltas = []
for init_time in mpas_inits:
    valid_nearest = valid_times_da.sel(t=init_time, method="nearest")
    if np.abs(valid_nearest.values - init_time) <= pd.Timedelta(*tolerance):
        valid_mpas_inits.append(init_time)
        valid_valid_inits.append(valid_nearest)
        deltas.append(init_time - valid_nearest.values)

In [21]:
deltas

[Timedelta('0 days 00:04:52.541241984'),
 Timedelta('0 days 00:04:52.226861056'),
 Timedelta('-1 days +23:54:53.758082944'),
 Timedelta('0 days 00:04:53.510736896'),
 Timedelta('0 days 00:04:53.382849024'),
 Timedelta('0 days 00:04:53.699975040'),
 Timedelta('0 days 00:04:53.680663040'),
 Timedelta('0 days 00:04:53.375324032'),
 Timedelta('0 days 00:04:53.362638976'),
 Timedelta('0 days 00:04:53.596163968'),
 Timedelta('0 days 00:04:53.577133952'),
 Timedelta('0 days 00:04:53.309659904'),
 Timedelta('0 days 00:04:53.285803008'),
 Timedelta('0 days 00:04:53.441957888'),
 Timedelta('0 days 00:04:53.470636032'),
 Timedelta('0 days 00:04:53.219911040'),
 Timedelta('0 days 00:04:53.177278976'),
 Timedelta('0 days 00:04:53.314982016'),
 Timedelta('0 days 00:04:53.395277952'),
 Timedelta('0 days 00:04:53.134774016'),
 Timedelta('0 days 00:04:53.104056064'),
 Timedelta('0 days 00:04:53.317302912'),
 Timedelta('0 days 00:04:53.306697984'),
 Timedelta('0 days 00:04:52.996929024'),
 Timedelta('0 